<a href="https://colab.research.google.com/github/SeoulTechPSE/tong_lectures/blob/main/5_gravity_cosmo_strings/cosmology/cosmo_ch02_figures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Cosmology* — `cosmo_ch02_figures.ipynb` · after D. Tong's Cambridge lecture notes · Kee-Youn Yoo, Process Systems Laboratory, SeoulTech

# Cosmology Ch 2 — Figures

Generates all Python figures for `cosmo_ch02_slides.qmd`.
Output PNGs are saved to `figures/` (relative to the `.qmd` file).

| File | Slide | Content |
|------|-------|---------|
| `cosmo_maxwell_boltzmann.png` | §2.1.2 | Maxwell-Boltzmann speed distribution |
| `cosmo_planck_bb.png` | §2.2.1 | Planck blackbody spectrum (several T) |
| `cosmo_saha.png` | §2.3.3 | Ionisation fraction $X_e$ vs $T$ |
| `cosmo_gstar.png` | §2.4.2 | Effective d.o.f. $g_\star$ vs $k_BT$ |
| `cosmo_bbn.png` | §2.5.3 | BBN: $n_n/n_p$ & $^4$He yield vs time |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.special import zeta
from scipy.optimize import brentq
import os

os.makedirs('figures', exist_ok=True)

# ── colour palette (matches cosmo_slides.css) ──────────────────
BG      = '#080c14'
SURFACE = '#0f1825'
BORDER  = '#1e2d42'
TEXT    = '#e2eaf4'
MUTED   = '#7a9ab8'
GOLD    = '#e8b84b'
BLUE    = '#4ca8f0'
GREEN   = '#3ecf8e'
RED     = '#f06060'
PURPLE  = '#c084fc'
ORANGE  = '#f59e42'

def style_ax(ax):
    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values():
        spine.set_color(BORDER)
    ax.tick_params(colors=MUTED, labelsize=10)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.title.set_color(GOLD)
    ax.grid(True, color=BORDER, linewidth=0.6, alpha=0.7)

print('Setup complete.')

## Fig 1 — Maxwell-Boltzmann Speed Distribution

In [ ]:
from scipy.constants import k as kB, u   # Boltzmann constant, atomic mass unit

T = 298.0   # 25 °C  (matches Tong Fig 27)

# masses in kg
gases = [
    ('$^4$He',   4*u,  BLUE),
    ('$^{20}$Ne',20*u, GREEN),
    ('$^{40}$Ar',40*u, GOLD),
    ('$^{132}$Xe',132*u, RED),
]

v = np.linspace(0, 3000, 3000)  # m/s

fig, ax = plt.subplots(figsize=(7, 4), facecolor=BG)
style_ax(ax)

for name, m, col in gases:
    fv = 4*np.pi * v**2 * (m/(2*np.pi*kB*T))**1.5 * np.exp(-m*v**2/(2*kB*T))
    ax.plot(v, fv*1e-3, color=col, lw=2.0, label=name)  # ×1e-3 → s/m per (m/s)

ax.set_xlabel('Speed  $v$  (m/s)', fontsize=12)
ax.set_ylabel('Probability density  (s/m)  ×10$^{-3}$', fontsize=11)
ax.set_title('Maxwell-Boltzmann Distribution  ($T = 25$°C)', fontsize=13, pad=8)
ax.set_xlim(0, 2600)
ax.set_ylim(0, None)
ax.legend(framealpha=0.25, facecolor=SURFACE, edgecolor=BORDER, fontsize=11)

plt.tight_layout(pad=0.4)
plt.savefig('figures/cosmo_maxwell_boltzmann.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Fig 1 saved.')

## Fig 2 — Planck Blackbody Spectrum

In [ ]:
from scipy.constants import h, c, k as kB

# Spectral energy density B_lambda (W/m^2/sr/m) vs wavelength
lam = np.linspace(0.1e-6, 3.0e-6, 3000)  # 0.1 – 3 µm

temps = [3000, 4000, 5000, 6000]
colors = [RED, ORANGE, GOLD, BLUE]

# Visible band roughly 380–700 nm
vis_lo, vis_hi = 380e-9, 700e-9

fig, ax = plt.subplots(figsize=(7.5, 4.5), facecolor=BG)
style_ax(ax)

for T, col in zip(temps, colors):
    B = (2*h*c**2/lam**5) / (np.exp(h*c/(lam*kB*T)) - 1)
    B_norm = B / B.max()        # normalise to peak=1 for visual clarity
    ax.plot(lam*1e6, B_norm, color=col, lw=2.2, label=f'$T = {T}$ K')
    # mark peak (Wien's law: λ_max = b/T, b=2.898e-3 m·K)
    lam_peak = 2.898e-3 / T
    ax.axvline(lam_peak*1e6, color=col, lw=0.8, ls=':', alpha=0.6)

# shade visible band
ax.axvspan(vis_lo*1e6, vis_hi*1e6, alpha=0.08, color=BLUE, label='Visible')
ax.text(0.54, 0.92, 'visible', transform=ax.transAxes,
        color=BLUE, fontsize=9, alpha=0.8, ha='center')

ax.set_xlabel('Wavelength  $\\lambda$  (µm)', fontsize=12)
ax.set_ylabel('Intensity (normalised)', fontsize=12)
ax.set_title('Planck Blackbody Spectrum', fontsize=13, pad=8)
ax.set_xlim(0.1, 3.0)
ax.set_ylim(0, 1.12)
ax.legend(framealpha=0.25, facecolor=SURFACE, edgecolor=BORDER, fontsize=11)

# UV / IR labels
ax.text(0.22, 1.05, 'UV', color=MUTED, fontsize=9, ha='center')
ax.text(2.5,  1.05, 'IR', color=MUTED, fontsize=9, ha='center')

plt.tight_layout(pad=0.4)
plt.savefig('figures/cosmo_planck_bb.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Fig 2 saved.')

## Fig 3 — Saha Equation: Ionisation Fraction $X_e(T)$

In [ ]:
from scipy.constants import k as kB, hbar, m_e, c, physical_constants

# Physical constants
E_bind  = 13.6 * 1.602e-19    # J  (binding energy of hydrogen)
eta     = 6e-10                # baryon-to-photon ratio (≈ 5×10^-10 from Planck)
zeta3   = zeta(3)              # Riemann zeta(3) ≈ 1.202

# Temperature range: 500 K to 10000 K
T_arr = np.linspace(500, 10000, 5000)

def Xe_saha(T):
    """Solve Saha equation for X_e at temperature T."""
    beta = 1.0 / (kB * T)
    # n_gamma = (2 zeta(3) / pi^2) * (kBT / hbar c)^3
    n_gamma = (2*zeta3/np.pi**2) * (kB*T / (hbar*c))**3
    # RHS of Saha: (1-Xe)/Xe^2 = rhs
    # rhs = eta * n_gamma * (2*pi*hbar^2 / (m_e*kBT))^(3/2) * exp(beta*E_bind)
    Lambda3 = (2*np.pi*hbar**2 / (m_e*kB*T))**1.5
    rhs = eta * n_gamma * Lambda3 * np.exp(beta * E_bind)
    # Solve quadratic: rhs * Xe^2 + Xe - 1 = 0
    # Xe = (-1 + sqrt(1 + 4*rhs)) / (2*rhs)
    Xe = (-1 + np.sqrt(1 + 4*rhs)) / (2*rhs)
    return np.clip(Xe, 0, 1)

Xe_arr = np.array([Xe_saha(T) for T in T_arr])

# Find T_rec (Xe = 0.5) and T_last (Xe = 0.1)
T_half  = T_arr[np.argmin(np.abs(Xe_arr - 0.5))]
T_tenth = T_arr[np.argmin(np.abs(Xe_arr - 0.1))]
print(f'X_e = 0.5  at T ≈ {T_half:.0f} K')
print(f'X_e = 0.1  at T ≈ {T_tenth:.0f} K  (T_rec in Tong)')

fig, ax = plt.subplots(figsize=(7.5, 4.5), facecolor=BG)
style_ax(ax)

ax.plot(T_arr, Xe_arr, color=GOLD, lw=2.5, label='$X_e(T)$ — Saha')

# Mark X_e = 0.1 (Tong's benchmark)
ax.axhline(0.1, color=MUTED, lw=1.0, ls='--', alpha=0.7)
ax.axvline(T_tenth, color=RED, lw=1.2, ls='--', alpha=0.8)
ax.text(T_tenth+120, 0.13, f'$T_{{\\rm rec}}\\approx{T_tenth:.0f}$ K\n($X_e=0.1$)',
        color=RED, fontsize=9.5)

# Mark X_e = 0.5
ax.axvline(T_half, color=BLUE, lw=1.0, ls=':', alpha=0.6)
ax.text(T_half+100, 0.53, f'{T_half:.0f} K', color=BLUE, fontsize=9)

# Shading
ax.fill_between(T_arr, Xe_arr, 0, alpha=0.08, color=GOLD)

ax.set_xlabel('Temperature  $T$  (K)', fontsize=12)
ax.set_ylabel('Ionisation fraction  $X_e$', fontsize=12)
ax.set_title('Recombination: Saha Equation', fontsize=13, pad=8)
ax.set_xlim(500, 10000)
ax.set_ylim(-0.02, 1.06)

# Top x-axis: redshift z = T/T0 - 1  (T0 = 2.726 K)
T0 = 2.726
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
z_ticks = np.array([200, 500, 1000, 1500, 2000, 3000])
T_ticks = T0 * (1 + z_ticks)
ax2.set_xticks(T_ticks)
ax2.set_xticklabels([str(z) for z in z_ticks], fontsize=9)
ax2.tick_params(colors=MUTED)
ax2.set_xlabel('Redshift  $z$', fontsize=11, color=TEXT)
ax2.spines['top'].set_color(BORDER)

ax.legend(framealpha=0.25, facecolor=SURFACE, edgecolor=BORDER, fontsize=11)
plt.tight_layout(pad=0.4)
plt.savefig('figures/cosmo_saha.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Fig 3 saved.')

## Fig 4 — Effective Relativistic Degrees of Freedom $g_\star(T)$

In [ ]:
# Standard Model particle content and mass thresholds
# Each entry: (name, kBT_threshold in MeV, delta_g_boson, delta_g_fermion)
# g_star = sum_bosons g_i + (7/8) sum_fermions g_i
# Spin degeneracy: photon=2, W±/Z=3each, Higgs=1, gluons=2×8=16
#                 e/mu/tau=2each+antiparticle=4, quarks=2×3colors×6flavors×2=72
#                 neutrinos=1each×3gen×2(nu+nubar)=6

# We build g_star as a step function dropping at each threshold
# Thresholds (kBT where particle becomes non-relativistic ≈ mc^2)
SM_particles = [
    # (label, kBT_threshold MeV, g_boson, g_fermion)
    ('$e^\\pm$',          0.511,         0,  4),   # electron + positron
    ('$\\mu^\\pm$',       105.7,         0,  4),   # muon
    ('$\\pi^{0,\\pm}$',   135.0,         3,  0),   # pions (pseudo-Goldstone, approx boson)
    ('$\\tau^\\pm$',      1777,          0,  4),   # tau
    ('$s\\bar s$',        95,            0, 12),   # strange quark (QCD scale ~150 MeV)
    ('QCD / $u,d,s$ quarks+gluons', 150, 16+2, 12+12+12),  # QCD transition
    ('$c\\bar c$',        1270,          0, 12),
    ('$b\\bar b$',        4180,          0, 12),
    ('$W^\\pm,Z$',        80400,         9,  0),
    ('Higgs',            125000,         1,  0),
    ('$t\\bar t$',       173000,         0, 12),
]

# Build g_star(T) piecewise
# Start from high T (full SM) and go down
# Simpler: enumerate key epochs
epochs = [
    # (kBT_MeV_hi, kBT_MeV_lo, g_star)
    (1e6,    1e5,  106.75),  # full SM (all particles relativistic)
    (1e5,    1e4,   96.25),  # top quark gone
    (1e4,    1e3,   86.25),  # Higgs, W, Z gone
    (1e3,    2e2,   75.75),  # b quark gone
    (2e2,    1.5e2, 61.75),  # c quark, tau gone
    (1.5e2,  1e-1,  17.25),  # QCD transition: quarks/gluons → hadrons; mostly π, then disappear
    (1e-1,   5.11e-1, 10.75),# before e+e- annihilation
    (5.11e-1,1e-2,  10.75),  # e+e- still relativistic
    (1e-2,   1e-4,   3.91),  # after e+e- annihilation (photons + neutrinos)
    (1e-4,   1e-6,   3.36),  # neutrinos slightly cooler
]

# Simplified but physically accurate step function
kBT_vals = np.logspace(-4, 6, 2000)  # MeV

def gstar(kBT_MeV):
    """Simplified g_star(T) for Standard Model."""
    kT = kBT_MeV
    if kT > 1e5:
        return 106.75
    elif kT > 8e4:
        return 96.25   # top quark decouples ~173 GeV → just below
    elif kT > 1.7e4:
        return 86.25   # Higgs + W + Z
    elif kT > 4.2e3:
        return 75.75   # b quark
    elif kT > 1.3e3:
        return 61.75   # charm + tau
    elif kT > 1.5e2:
        return 21.25   # QCD transition (pions, eta still around)
    elif kT > 1e-1:
        return 10.75   # photons + 3nu + e±
    elif kT > 5e-4:
        return 3.91    # photons + 3nu (with T_nu = (4/11)^(1/3) T_gamma)
    else:
        return 3.36

gstar_arr = np.array([gstar(kT) for kT in kBT_vals])

fig, ax = plt.subplots(figsize=(8, 4.5), facecolor=BG)
style_ax(ax)

ax.step(kBT_vals, gstar_arr, where='post', color=GOLD, lw=2.5)
ax.fill_between(kBT_vals, gstar_arr, step='post', alpha=0.12, color=GOLD)

# Key annotations
annotations = [
    (3e5,  110, '$g_\\star = 106.75$\n(full SM)', BLUE),
    (3,    12.5, '$g_\\star = 10.75$\n($\\gamma+3\\nu+e^\\pm$)', GREEN),
    (5e-3, 5.0, '$g_\\star \\approx 3.4$\n(after $e^\\pm$ annihilation)', ORANGE),
]
for x, y, txt, col in annotations:
    ax.text(x, y, txt, color=col, fontsize=8.5, va='center', ha='center',
            bbox=dict(facecolor=SURFACE, edgecolor=BORDER, boxstyle='round,pad=0.3', alpha=0.85))

# Mark QCD transition
ax.axvline(150, color=RED, lw=1.2, ls='--', alpha=0.7)
ax.text(160, 55, 'QCD\ntransition\n150 MeV', color=RED, fontsize=8.5)

# Mark EW transition
ax.axvline(1e5, color=PURPLE, lw=1.2, ls='--', alpha=0.7)
ax.text(1.1e5, 50, 'EW\n100 GeV', color=PURPLE, fontsize=8.5)

ax.set_xscale('log')
ax.set_xlabel('$k_BT$  (MeV)', fontsize=12)
ax.set_ylabel('$g_\\star(T)$', fontsize=13)
ax.set_title('Effective Relativistic Degrees of Freedom', fontsize=13, pad=8)
ax.set_xlim(1e-4, 1e6)
ax.set_ylim(0, 120)

# Second x-axis: temperature in K
from scipy.constants import k as kB_SI, eV
ax2 = ax.twiny()
ax2.set_xscale('log')
ax2.set_xlim(ax.get_xlim())
# kBT [MeV] → T [K]:  T = kBT * 1e6*eV / kB_SI
T_for_ticks = np.array([1e7, 1e9, 1e11, 1e13, 1e15])  # K
kBT_for_ticks = T_for_ticks * kB_SI / (1e6 * eV)       # MeV
ax2.set_xticks(kBT_for_ticks)
ax2.set_xticklabels([f'$10^{{{int(np.log10(T))}}}$' for T in T_for_ticks], fontsize=9)
ax2.set_xlabel('Temperature  $T$  (K)', fontsize=11, color=TEXT)
ax2.tick_params(colors=MUTED)
ax2.spines['top'].set_color(BORDER)

plt.tight_layout(pad=0.4)
plt.savefig('figures/cosmo_gstar.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Fig 4 saved.')

## Fig 5 — Big Bang Nucleosynthesis: $n_n/n_p$ and $^4$He yield

In [ ]:
from scipy.constants import k as kB_SI, eV

# Physical parameters
Delta_m_c2 = 1.293   # MeV  (neutron-proton mass difference)
tau_n      = 880.2   # s    (neutron lifetime)
kBT_dec    = 0.80    # MeV  (neutron freeze-out temperature)
t_dec      = 2.0     # s    (freeze-out time)
kBT_deut   = 0.063   # MeV  (deuterium formation temperature)
t_deut     = 360.0   # s    (deuterium bottleneck overcome)
kBT_bbn_end= 0.030   # MeV  (BBN essentially complete)
t_bbn_end  = 1000.0  # s

# Time axis: 0.1 s to 1200 s (log scale)
t = np.logspace(-1, 3.1, 3000)  # seconds

# Temperature vs time (using g_star = 10.75 for radiation era):
# kBT [MeV] = (2.4/g_star^0.5) * (1 s/t)^0.5   → reversed
# t [s] = (2.4/g_star^0.5)^2 / (kBT[MeV])^2
gstar_bbn = 10.75
kBT_of_t = (2.4 / np.sqrt(gstar_bbn)) / np.sqrt(t)  # MeV

# ── Neutron-to-proton ratio ─────────────────────────────────────
# Phase 1: t < t_dec  → equilibrium ratio
# Phase 2: t_dec < t < t_deut → freeze-out value × neutron decay
# Phase 3: t > t_deut → rapid He formation; neutrons bound

ratio_eq   = np.exp(-Delta_m_c2 / kBT_of_t)   # equilibrium: exp(-Δmc²/kBT)
ratio_dec  = np.exp(-Delta_m_c2 / kBT_dec)    # value at freeze-out ≈ 1/5

nn_np = np.where(
    t <= t_dec,
    ratio_eq,
    np.where(
        t <= t_deut,
        ratio_dec * np.exp(-(t - t_dec) / tau_n),   # free neutron decay
        ratio_dec * np.exp(-(t_deut - t_dec) / tau_n)  # bound in He (constant)
    )
)
nn_np = np.clip(nn_np, 0, 1)

# ── Helium-4 mass fraction Y_p ───────────────────────────────────
# Before deuterium forms: Y_p = 0 (neutrons still free / in decay)
# After: all neutrons rapidly bind → Y_p = 2*(nn/np) / (1 + nn/np) * 2
#  more precisely: Y_p = 4*(nn/2) / (nn + np) = 2*(nn/np)/(1 + nn/np)
ratio_at_deut = ratio_dec * np.exp(-(t_deut - t_dec) / tau_n)
Yp_final = 2 * ratio_at_deut / (1 + ratio_at_deut)
print(f'n_n/n_p at freeze-out : {ratio_dec:.3f}  (≈ 1/5)')
print(f'n_n/n_p at deuterium  : {ratio_at_deut:.3f}  (≈ 1/7)')
print(f'Yp (He-4 mass fraction): {Yp_final:.3f}  (≈ 25%)')

Yp = np.where(t < t_deut, 0.0, Yp_final)

# ── Plot ──────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(8.5, 4.8), facecolor=BG)
style_ax(ax1)

# n_n/n_p ratio
ln1, = ax1.plot(t, nn_np, color=BLUE, lw=2.5, label='$n_n/n_p$')
ax1.set_ylabel('$n_n / n_p$', fontsize=12, color=BLUE)
ax1.tick_params(axis='y', colors=BLUE)
ax1.set_ylim(0, 0.55)

# He-4 mass fraction (right axis)
ax2 = ax1.twinx()
ax2.set_facecolor('none')
ln2, = ax2.plot(t, Yp, color=GOLD, lw=2.5, ls='--', label='$Y_p$ ($^4$He mass fraction)')
ax2.set_ylabel('$Y_p$ (He-4 mass fraction)', fontsize=12, color=GOLD)
ax2.tick_params(axis='y', colors=GOLD)
ax2.set_ylim(0, 0.55)
ax2.spines['right'].set_color(BORDER)

# Mark key events
ax1.axvline(t_dec, color=GREEN, lw=1.5, ls='--', alpha=0.8)
ax1.text(t_dec*1.05, 0.47, 'neutron\nfreeze-out\n$t\\approx2$ s', color=GREEN, fontsize=8.5)

ax1.axvline(t_deut, color=ORANGE, lw=1.5, ls='--', alpha=0.8)
ax1.text(t_deut*1.06, 0.32, 'deuterium\nbottleneck\n$t\\approx360$ s', color=ORANGE, fontsize=8.5)

# Observed Yp
ax2.axhline(0.245, color=RED, lw=1.2, ls=':', alpha=0.7)
ax2.text(1.2, 0.255, 'observed $Y_p\\approx 24.5\\%$', color=RED, fontsize=8.5)

ax1.set_xscale('log')
ax1.set_xlabel('Time  $t$  (seconds)', fontsize=12)
ax1.set_title('Big Bang Nucleosynthesis', fontsize=13, pad=8)
ax1.set_xlim(0.1, 1200)

# Top axis: temperature
ax3 = ax1.twiny()
ax3.set_xscale('log')
ax3.set_xlim(ax1.get_xlim())
t_ticks = np.array([0.1, 0.3, 1, 3, 10, 100, 1000])
kBT_ticks = (2.4/np.sqrt(gstar_bbn)) / np.sqrt(t_ticks)
ax3.set_xticks(t_ticks)
ax3.set_xticklabels([f'{v:.2g}' for v in kBT_ticks], fontsize=9)
ax3.set_xlabel('$k_BT$  (MeV)', fontsize=11, color=TEXT)
ax3.tick_params(colors=MUTED)
ax3.spines['top'].set_color(BORDER)

lines = [ln1, ln2]
labs  = [l.get_label() for l in lines]
ax1.legend(lines, labs, framealpha=0.25, facecolor=SURFACE, edgecolor=BORDER, fontsize=11,
           loc='center left')

plt.tight_layout(pad=0.4)
plt.savefig('figures/cosmo_bbn.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Fig 5 saved.')
print(f'\nAll 5 figures saved to figures/')